# VALT Summer School — Mapping & SQANTI-reads of long-read RNA-seq

**Session:** July 25, evening — *Mapping practical + SQANTI-reads*

**Data:** human **chromosome 8**, **H1 PacBio Iso-Seq** (full-length cDNA reads) — the same
reads used in tomorrow's IsoTools and SQANTI3 sessions. **6 samples in 2 groups:**
**`h1`** (undifferentiated H1 embryonic stem cells) vs **`endo`** (H1-derived endoderm),
3 replicates each.

### What you will do
1. **Map** long (Iso-Seq) reads to the genome with `minimap2` and inspect the alignments.
2. **Classify** every read against the reference annotation with **SQANTI3 QC** (structural categories).
3. **Aggregate** the 6 samples with **SQANTI-reads** — faceted by group (`endo` vs `h1`) — into
   QC tables + a multi-sample report, and **interpret** library quality and annotation completeness.

> **Cells marked `# TODO`** are for you to complete (hints in the comments). Run top-to-bottom.
> **Environment — pick the `sqanti3` env** (`.conda/envs/sqanti3`, Python 3.11):
> in **VS Code / Cursor** choose **`sqanti3`** from the interpreter list (⚠️ *not* the similarly-named
> **`SQANTI3.env`**, even though it says *Recommended* — it is incomplete); in **JupyterLab** choose
> the **Python (SQANTI3)** kernel.

### Background: why "SQANTI-reads"?
Classic SQANTI3 characterizes a *transcriptome assembly*. **SQANTI-reads** runs the SQANTI3
structural classification on the **reads themselves**, fingerprints each read's splice pattern as a
hashed **UJC** (Unique Junction Chain), and compares many samples/groups — to assess data quality
and spot genes whose annotation looks incomplete. (Keil, Monzó, McIntyre & Conesa, *Genome Res* 2025.)


## 0. Setup

This is **your own copy** of the practical folder. `data/` holds the per-sample reads
(`data/*.fastq`) and the chr8 `reference/`. Everything you generate (BAMs, SQANTI outputs)
is written here in your folder, so nothing collides with other students.

In [ ]:
import os, sys, glob, subprocess, textwrap
import pandas as pd

# The 'Python (SQANTI3)' kernel runs the env's python but does NOT put the env's
# bin/ on PATH, so shell-escape (!) commands can't find minimap2/samtools/sqanti3_*.
# Prepend it here so `!minimap2 ...` etc. work in this notebook.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", "")

# Threads per step — keep modest: the whole class shares this machine.
THREADS = 4

DATA    = "data"                       # per-sample reads (data/*.fastq) + reference/
REF_FA  = f"{DATA}/reference/GRCh38.p14.chr8.fa"
REF_GTF = f"{DATA}/reference/gencode.v45.chr8.gtf"

# 6 samples, 2 groups (the design for SQANTI-reads)
GROUP = {"endo_1":"endo","endo_2":"endo","endo_3":"endo","h1_1":"h1","h1_2":"h1","h1_3":"h1"}
SAMPLES = list(GROUP)

print("Reference :", REF_FA)
for s in SAMPLES:
    n = sum(1 for _ in open(f"{DATA}/{s}.fastq")) // 4
    print(f"  {s:7s} [{GROUP[s]:4s}] : {n:,} reads")


In [ ]:
# Confirm the tools are on PATH (sqanti3 env)
!which minimap2 samtools sqanti3_qc.py sqanti3_reads.py
!minimap2 --version

## Part 1 — Mapping practical

PacBio Iso-Seq reads are **full-length cDNA** spanning multiple exons, so the aligner must be
**splice-aware** (it places long gaps = introns). `minimap2` has a preset for high-quality spliced
cDNA: `-ax splice:hq`; `-uf` tells it transcripts come from the **forward** mRNA strand.

> **Note on the data:** these reads were pre-selected to chr8, so you'll see ~100% mapping here.
> With raw genome-wide reads, mapping rate is one of the first QC numbers you'd inspect.


In [ ]:
# Map one sample (endo_1) to chr8 with minimap2, then sort + index.
!minimap2 -t {THREADS} -ax splice:hq -uf {REF_FA} data/endo_1.fastq | samtools sort -@ {THREADS} -o endo_1.chr8.bam -
!samtools index endo_1.chr8.bam
print("done")

In [ ]:
# Alignment summary
!samtools flagstat endo_1.chr8.bam

### Look at a spliced alignment
In the CIGAR string, **`N`** = a skipped reference region, i.e. an **intron**. Long `N` operations
are the splice junctions the long read crosses.

In [ ]:
import pysam
bam = pysam.AlignmentFile("endo_1.chr8.bam", "rb")
for r in bam.fetch():
    if not r.is_secondary and not r.is_supplementary and r.cigarstring and "N" in r.cigarstring:
        n_introns = sum(1 for op, _ in r.cigartuples if op == 3)  # 3 = N
        print(f"read {r.query_name}")
        print(f"  pos chr8:{r.reference_start:,}  introns(N)={n_introns}")
        print(f"  CIGAR (first 90 chars): {r.cigarstring[:90]}")
        break
bam.close()

**Q1.** What fraction of reads are *primary* vs *secondary/supplementary*? What does a
secondary alignment mean for a full-length transcript read?
**Q2.** Roughly how many introns does the example read cross? How would mono-exon reads look?

## Part 2 — SQANTI3 QC + SQANTI-reads aggregation

`sqanti3_reads.py`:
1. runs **SQANTI3 QC** on each sample (maps reads with minimap2 and classifies every read into a
   **structural category** — FSM, ISM, NIC, NNC, … — relative to the reference annotation),
2. computes a **UJC** (hashed splice-chain) per read,
3. **aggregates** all samples into QC tables and a report.

It is driven by a **design file**. Ours has a **`group`** column (`endo` vs `h1`); passing
`--factor group` makes SQANTI-reads facet the plots by group so we can compare the two cell states.

> ⏱ **The next cell takes ~20 minutes** — it maps and classifies every read across all 6 samples.
> Start it, then read on / work through Part 1 while it runs.

In [ ]:
# Design file: sampleID, file_acc (= FASTQ name prefix), and the group factor.
rows = "sampleID,file_acc,group\n" + "\n".join(f"{s},{s},{GROUP[s]}" for s in SAMPLES) + "\n"
open("design.csv", "w").write(rows)
print(rows)

In [ ]:
# Run SQANTI-reads end-to-end on all 6 samples, faceted by group (~20 minutes).
!sqanti3_reads.py \
    --refFasta {REF_FA} --refGTF {REF_GTF} \
    -de design.csv -i data -o out \
    -f group -t {THREADS} --report pdf --all_tables
print("SQANTI-reads finished")

In [ ]:
print("Per-sample SQANTI3 output (endo_1):")
!ls out/endo_1 | head
print("\nAggregated SQANTI-reads tables + report:")
!ls -1 out/*.csv out/*.pdf 2>/dev/null

## Part 3 — Explore the QC tables

SQANTI-reads writes one row per sample (or per gene/UJC). We load them with pandas
(using `glob`, so the code is robust to the output prefix).

In [ ]:
def load(pattern):
    hits = glob.glob(f"out/*{pattern}*.csv")
    assert hits, f"no table matching {pattern}"
    return pd.read_csv(hits[0])

gene   = load("gene_counts")
ujc    = load("ujc_counts")
length = load("length_summary")
cv     = load("cv")
print("gene_counts   :", gene.shape)
print("ujc_counts    :", ujc.shape)
print("length_summary:", length.shape)
print("cv            :", cv.shape)
length

### Structural categories per sample (coloured intuition: endo vs h1)
The mix of categories is a quick read on library quality. Compare the two groups.

In [ ]:
frames = []
for s in SAMPLES:
    cls = pd.read_csv(f"out/{s}/{s}_classification.txt", sep="\t", usecols=["structural_category"])
    vc = cls["structural_category"].value_counts(); vc.name = s
    frames.append(vc)
cat = pd.concat(frames, axis=1).fillna(0).astype(int)[SAMPLES]
display(cat)
ax = (100*cat/cat.sum()).T.plot(kind="bar", stacked=True, figsize=(9,4))
ax.set_ylabel("% of reads"); ax.set_title("Structural categories per sample (endo_* vs h1_*)")
ax.legend(bbox_to_anchor=(1.0, 1.0));

### Read length per sample
`length_summary` summarises read lengths per sample (`median_length`, `% reads > 1kb`, …).

In [ ]:
col   = [c for c in length.columns if "median" in c.lower()][0]
idcol = [c for c in length.columns if c.lower() in ("sampleid","sample","id")][0]
length.plot(x=idcol, y=col, kind="bar", legend=False, title="Median read length per sample");

## Part 4 — Under-annotation & the multi-sample report

SQANTI-reads flags genes that are well-covered by reads but whose reference annotation may be
**incomplete** (a highly-supported novel splice chain with no matching annotated transcript).

In [ ]:
ua = None
for pat in ["gene_class", "underannot"]:
    hits = glob.glob(f"out/*{pat}*.csv")
    if hits:
        ua = pd.read_csv(hits[0]); print("loaded", hits[0]); break
if ua is not None:
    display(ua.head())
    catcol = [c for c in ua.columns if "categ" in c.lower() or "class" in c.lower()]
    if catcol:
        display(ua[catcol[-1]].value_counts())

### The report (faceted by group)
SQANTI-reads renders all metrics into a PDF — structural categories, UJC counts, read length,
junction CV (donors/acceptors), a cross-sample **PCA**, and the under-annotation summary — with
**`endo` vs `h1`** shown side by side. We render the first pages inline.

In [ ]:
from pdf2image import convert_from_path
allpdfs = glob.glob("out/*plots*.pdf")
main = [p for p in allpdfs if "annotation" not in os.path.basename(p)]
report = (main or allpdfs)[0]
print("main report:", report)
for pg in convert_from_path(report, dpi=80, first_page=1, last_page=4):
    display(pg)

### Discussion
- **Q3.** In the PCA / category mix, do `endo` and `h1` separate? Do replicates cluster within group?
  If one replicate were an outlier, what would you check first?
- **Q4.** Pick a gene flagged `underannotated_with_candidate_transcript`. What evidence supports a
  *missing* transcript there, and how would you confirm it?
- **Q5.** Why is the UJC (hashed splice chain) a better unit than raw read counts for comparing
  isoform structures across samples?

---
**Recap:** you mapped long reads (minimap2, splice-aware) → classified reads into structural
categories (SQANTI3 QC) → aggregated 6 samples in 2 groups into UJC/length/CV/PCA metrics and an
under-annotation report (SQANTI-reads), comparing **H1 vs endoderm** library quality and annotation
completeness. Tomorrow you'll assemble these same reads (IsoTools) and characterize the assembly (SQANTI3).